# Extract Pauli Error Rates from FakeFez Noise Model

This notebook extracts ground truth Pauli error rates directly from the noise model,
rather than measuring them empirically through circuit simulation.

**Advantages:**
- No statistical noise (exact values)
- No contamination from state prep/measurement gates
- Can compute composed errors for any path
- Much faster than running simulations

In [1]:
import numpy as np
from collections import defaultdict
from typing import List, Tuple, Dict
import pickle
import os
import networkx as nx

from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit_aer.noise import NoiseModel
from qiskit.quantum_info import Kraus, SuperOp, PTM

np.set_printoptions(precision=6, suppress=True)

## 1. Load Backend and Noise Model

In [2]:
backend = FakeFez()
noise_model = NoiseModel.from_backend(backend)
target = backend.target

print(f"Backend: {backend.name}")
print(f"Qubits: {backend.num_qubits}")
print(f"Local quantum errors for: {list(noise_model._local_quantum_errors.keys())}")

Backend: fake_fez
Qubits: 156
Local quantum errors for: ['sx', 'id', 'x', 'reset', 'cz']


## 2. Functions to Extract Pauli Rates from Quantum Errors

In [3]:
def get_quantum_error(noise_model, gate_name, qubits):
    """
    Get QuantumError for a specific gate and qubit(s).
    
    Args:
        noise_model: NoiseModel
        gate_name: 'sx', 'cz', etc.
        qubits: tuple, e.g., (0,) or (0, 1)
    """
    if gate_name in noise_model._local_quantum_errors:
        local = noise_model._local_quantum_errors[gate_name]
        if qubits in local:
            return local[qubits]
    if gate_name in noise_model._default_quantum_errors:
        return noise_model._default_quantum_errors[gate_name]
    return None


def quantum_error_to_pauli_rates_1q(quantum_error):
    """
    Convert a single-qubit QuantumError to Pauli error rates using PTM.
    
    For a Pauli channel, PTM diagonal is:
    [1, 1-2(py+pz), 1-2(px+pz), 1-2(px+py)]
    
    Returns:
        dict with 'pi', 'px', 'py', 'pz'
    """
    channel = quantum_error.to_quantumchannel()
    
    if channel.num_qubits != 1:
        raise ValueError("Expected single-qubit error")
    
    ptm = PTM(channel)
    ptm_matrix = ptm.data
    
    # Extract diagonal elements
    d1 = ptm_matrix[1, 1].real  # 1 - 2(py + pz)
    d2 = ptm_matrix[2, 2].real  # 1 - 2(px + pz)
    d3 = ptm_matrix[3, 3].real  # 1 - 2(px + py)
    
    # Solve for individual rates
    total_error = (3 - d1 - d2 - d3) / 4
    py_pz = (1 - d1) / 2
    px_pz = (1 - d2) / 2
    px_py = (1 - d3) / 2
    
    px = max(0, total_error - py_pz)
    py = max(0, total_error - px_pz)
    pz = max(0, total_error - px_py)
    pi = max(0, 1 - total_error)
    
    return {'pi': pi, 'px': px, 'py': py, 'pz': pz, 'total': px + py + pz}


def quantum_error_to_pauli_rates_2q(quantum_error):
    """
    Convert a two-qubit QuantumError to Pauli error rates.
    
    For two qubits, there are 16 Pauli operators: II, IX, IY, IZ, XI, ..., ZZ
    We use PTM and extract the diagonal to get depolarizing-like rates.
    
    Returns:
        dict with Pauli rates for each of the 16 operators
    """
    channel = quantum_error.to_quantumchannel()
    
    if channel.num_qubits != 2:
        raise ValueError("Expected two-qubit error")
    
    ptm = PTM(channel)
    ptm_matrix = ptm.data
    
    # Pauli order: II, IX, IY, IZ, XI, XX, XY, XZ, YI, YX, YY, YZ, ZI, ZX, ZY, ZZ
    paulis_2q = ['II', 'IX', 'IY', 'IZ', 'XI', 'XX', 'XY', 'XZ', 
                 'YI', 'YX', 'YY', 'YZ', 'ZI', 'ZX', 'ZY', 'ZZ']
    
    # Extract diagonal
    diag = np.diag(ptm_matrix).real
    
    # Compute total error from average shrinkage
    avg_shrinkage = np.mean(diag[1:])  # Exclude II
    total_error = (1 - avg_shrinkage) * 15 / 16
    
    # Approximate as depolarizing
    p_each = total_error / 15
    
    result = {
        'total_error': total_error,
        'p_depolarizing': p_each,
        'ptm_diagonal': diag,
        'paulis': paulis_2q
    }
    
    return result


print("Pauli extraction functions defined.")

Pauli extraction functions defined.


## 3. Extract Single-Qubit Gate Errors (SX)

In [4]:
# Extract Pauli rates for SX gate on all qubits
sx_pauli_rates = {}

for q in range(backend.num_qubits):
    sx_error = get_quantum_error(noise_model, 'sx', (q,))
    if sx_error:
        rates = quantum_error_to_pauli_rates_1q(sx_error)
        sx_pauli_rates[q] = rates

print(f"Extracted SX Pauli rates for {len(sx_pauli_rates)} qubits")

# Sample output
print("\nSample SX gate Pauli rates:")
for q in [0, 50, 100]:
    if q in sx_pauli_rates:
        r = sx_pauli_rates[q]
        print(f"  Qubit {q}: pX={r['px']:.6f}, pY={r['py']:.6f}, pZ={r['pz']:.6f}, total={r['total']:.6f}")

Extracted SX Pauli rates for 156 qubits

Sample SX gate Pauli rates:
  Qubit 0: pX=0.000370, pY=0.000370, pZ=0.000407, total=0.001146
  Qubit 50: pX=0.000091, pY=0.000091, pZ=0.000100, total=0.000281
  Qubit 100: pX=0.000039, pY=0.000039, pZ=0.000371, total=0.000449


## 4. Extract Two-Qubit Gate Errors (CZ)

In [5]:
# Extract Pauli rates for CZ gate on all edges
coupling_map = backend.coupling_map
cz_pauli_rates = {}

for edge in coupling_map.get_edges():
    cz_error = get_quantum_error(noise_model, 'cz', edge)
    if cz_error:
        rates = quantum_error_to_pauli_rates_2q(cz_error)
        edge_key = (min(edge), max(edge))  # Normalize to undirected
        if edge_key not in cz_pauli_rates:  # Avoid duplicates
            cz_pauli_rates[edge_key] = rates

print(f"Extracted CZ Pauli rates for {len(cz_pauli_rates)} edges")

# Sample output
print("\nSample CZ gate error rates:")
sample_edges = list(cz_pauli_rates.keys())[:5]
for edge in sample_edges:
    r = cz_pauli_rates[edge]
    print(f"  Edge {edge}: total_error={r['total_error']:.6f}, p_depol={r['p_depolarizing']:.6f}")

Extracted CZ Pauli rates for 176 edges

Sample CZ gate error rates:
  Edge (0, 1): total_error=0.012696, p_depol=0.000846
  Edge (1, 2): total_error=0.004715, p_depol=0.000314
  Edge (2, 3): total_error=0.003731, p_depol=0.000249
  Edge (3, 4): total_error=0.005437, p_depol=0.000362
  Edge (3, 16): total_error=0.005222, p_depol=0.000348


## 5. Compute Effective Pauli Channel for a SWAP Gate

A SWAP gate decomposes into 3 CNOTs, and CNOT decomposes into CZ + Hadamards.

- SWAP = CNOT(a,b) * CNOT(b,a) * CNOT(a,b)
- Each CNOT(a,b) = H(b) * CZ(a,b) * H(b)

So a SWAP involves:
- 3 CZ gates
- 6 H gates (decomposed to SX + RZ)

The total error accumulates from all these gates.

In [6]:
def estimate_swap_error(qubit_a, qubit_b, noise_model, target):
    """
    Estimate the effective Pauli error for a SWAP gate between two qubits.
    
    SWAP decomposes to 3 CNOTs, each CNOT = H*CZ*H
    
    We approximate by summing error rates (valid for small errors).
    """
    # Get CZ error for this edge
    edge = (qubit_a, qubit_b)
    edge_key = (min(edge), max(edge))
    
    cz_error = get_quantum_error(noise_model, 'cz', edge)
    if cz_error is None:
        cz_error = get_quantum_error(noise_model, 'cz', (edge[1], edge[0]))
    
    if cz_error:
        cz_rates = quantum_error_to_pauli_rates_2q(cz_error)
        cz_err = cz_rates['total_error']
    else:
        cz_err = 0.0
    
    # Get SX errors for both qubits
    sx_a = get_quantum_error(noise_model, 'sx', (qubit_a,))
    sx_b = get_quantum_error(noise_model, 'sx', (qubit_b,))
    
    sx_err_a = quantum_error_to_pauli_rates_1q(sx_a)['total'] if sx_a else 0.0
    sx_err_b = quantum_error_to_pauli_rates_1q(sx_b)['total'] if sx_b else 0.0
    
    # SWAP = 3 CNOTs, each CNOT uses 1 CZ and ~2 SX gates (for H decomposition)
    total_error = 3 * cz_err + 6 * (sx_err_a + sx_err_b)
    
    return {
        'total_error': total_error,
        'cz_error': cz_err,
        'sx_error_a': sx_err_a,
        'sx_error_b': sx_err_b,
        'edge': edge_key
    }


# Test on a sample edge
test_edge = list(coupling_map.get_edges())[0]
swap_err = estimate_swap_error(test_edge[0], test_edge[1], noise_model, target)

print(f"SWAP error estimate for edge {swap_err['edge']}:")
print(f"  CZ error (per gate):     {swap_err['cz_error']:.6f}")
print(f"  SX error (qubit {test_edge[0]}):    {swap_err['sx_error_a']:.6f}")
print(f"  SX error (qubit {test_edge[1]}):    {swap_err['sx_error_b']:.6f}")
print(f"  Total SWAP error:        {swap_err['total_error']:.6f}")

SWAP error estimate for edge (0, 1):
  CZ error (per gate):     0.012696
  SX error (qubit 0):    0.001146
  SX error (qubit 1):    0.000267
  Total SWAP error:        0.046569


## 6. Compute Average FakeFez Error Rates

Compute the average/median error rates across all FakeFez qubits and edges.

In [7]:
print("FakeFez Error Statistics")
print("=" * 60)

# Average single-qubit error
all_sx_errors = [r['total'] for r in sx_pauli_rates.values()]
avg_sx_error = np.mean(all_sx_errors)
median_sx_error = np.median(all_sx_errors)

# Average two-qubit error
all_cz_errors = [r['total_error'] for r in cz_pauli_rates.values()]
avg_cz_error = np.mean(all_cz_errors)
median_cz_error = np.median(all_cz_errors)

print(f"SX Gate Errors ({len(sx_pauli_rates)} qubits):")
print(f"  Mean:   {avg_sx_error:.6f}")
print(f"  Median: {median_sx_error:.6f}")
print(f"  Min:    {min(all_sx_errors):.6f}")
print(f"  Max:    {max(all_sx_errors):.6f}")

print(f"\nCZ Gate Errors ({len(cz_pauli_rates)} edges):")
print(f"  Mean:   {avg_cz_error:.6f}")
print(f"  Median: {median_cz_error:.6f} (more representative, excludes outliers)")
print(f"  Min:    {min(all_cz_errors):.6f}")
print(f"  Max:    {max(all_cz_errors):.6f}")

# For a single SWAP through a path qubit:
# Source data qubit -> Path qubit -> Sink data qubit = 2 SWAPs = 6 CZ + many SX
estimated_edge_error = 2 * (3 * median_cz_error + 6 * avg_sx_error)
print(f"\nEstimated error per logical edge (2 SWAPs): {estimated_edge_error:.6f}")
print(f"This is ~{estimated_edge_error*100:.2f}% total error per edge")

FakeFez Error Statistics
SX Gate Errors (156 qubits):
  Mean:   0.000474
  Median: 0.000361
  Min:    0.000150
  Max:    0.003732

CZ Gate Errors (176 edges):
  Mean:   0.046449
  Median: 0.004882 (more representative, excludes outliers)
  Min:    0.002674
  Max:    0.999938

Estimated error per logical edge (2 SWAPs): 0.034975
This is ~3.50% total error per edge


## 7. Extract Actual Pauli Rates (pX, pY, pZ) from Noise Model

Instead of assuming depolarizing (equal pX=pY=pZ), extract the actual distribution.

In [8]:
def compute_edge_pauli_ground_truth(sx_pauli_rates, cz_pauli_rates):
    """
    Compute Pauli error rates for a logical edge based on actual FakeFez noise.
    
    This version extracts the ACTUAL pX, pY, pZ distribution from the noise model
    rather than assuming depolarizing (equal) rates.
    
    Returns average rates across all qubits/edges.
    """
    # Collect actual Pauli rates from SX gates
    sx_px_list = [r['px'] for r in sx_pauli_rates.values()]
    sx_py_list = [r['py'] for r in sx_pauli_rates.values()]
    sx_pz_list = [r['pz'] for r in sx_pauli_rates.values()]
    
    avg_sx_px = np.mean(sx_px_list)
    avg_sx_py = np.mean(sx_py_list)
    avg_sx_pz = np.mean(sx_pz_list)
    avg_sx_total = avg_sx_px + avg_sx_py + avg_sx_pz
    
    # For CZ gates, extract marginal single-qubit Pauli rates from PTM
    cz_px_list = []
    cz_py_list = []
    cz_pz_list = []
    
    for edge, rates in cz_pauli_rates.items():
        diag = rates['ptm_diagonal']
        # Paulis order: II, IX, IY, IZ, XI, XX, XY, XZ, YI, YX, YY, YZ, ZI, ZX, ZY, ZZ
        # For qubit 0 (first position), X errors are: XI, XX, XY, XZ (indices 4-7)
        # For qubit 0, Y errors are: YI, YX, YY, YZ (indices 8-11)
        # For qubit 0, Z errors are: ZI, ZX, ZY, ZZ (indices 12-15)
        
        d_X = np.mean(diag[4:8])   # XI, XX, XY, XZ
        d_Y = np.mean(diag[8:12])  # YI, YX, YY, YZ
        d_Z = np.mean(diag[12:16]) # ZI, ZX, ZY, ZZ
        
        # Convert shrinkage to error rate: p = (1 - d) / 2
        px_0 = max(0, (1 - d_X) / 2)
        py_0 = max(0, (1 - d_Y) / 2)
        pz_0 = max(0, (1 - d_Z) / 2)
        
        cz_px_list.append(px_0)
        cz_py_list.append(py_0)
        cz_pz_list.append(pz_0)
    
    # Use median to avoid outliers
    median_cz_px = np.median(cz_px_list)
    median_cz_py = np.median(cz_py_list)
    median_cz_pz = np.median(cz_pz_list)
    
    print("Extracted Pauli rates from noise model:")
    print("=" * 60)
    print(f"SX gate (average across {len(sx_pauli_rates)} qubits):")
    print(f"  pX={avg_sx_px:.6f}, pY={avg_sx_py:.6f}, pZ={avg_sx_pz:.6f}")
    print(f"  Total: {avg_sx_total:.6f}")
    if avg_sx_total > 0:
        print(f"  Ratio pX:pY:pZ = {avg_sx_px/avg_sx_total:.2f}:{avg_sx_py/avg_sx_total:.2f}:{avg_sx_pz/avg_sx_total:.2f}")
    
    print(f"\nCZ gate (median across {len(cz_pauli_rates)} edges, single-qubit marginal):")
    median_cz_total = median_cz_px + median_cz_py + median_cz_pz
    print(f"  pX={median_cz_px:.6f}, pY={median_cz_py:.6f}, pZ={median_cz_pz:.6f}")
    print(f"  Total: {median_cz_total:.6f}")
    if median_cz_total > 0:
        print(f"  Ratio pX:pY:pZ = {median_cz_px/median_cz_total:.2f}:{median_cz_py/median_cz_total:.2f}:{median_cz_pz/median_cz_total:.2f}")
    
    # Compose for a SWAP (3 CZ + 6 SX per qubit, approximately)
    swap_px = 3 * median_cz_px + 6 * avg_sx_px
    swap_py = 3 * median_cz_py + 6 * avg_sx_py
    swap_pz = 3 * median_cz_pz + 6 * avg_sx_pz
    
    # Per edge = 2 SWAPs
    edge_px = 2 * swap_px
    edge_py = 2 * swap_py
    edge_pz = 2 * swap_pz
    edge_total = edge_px + edge_py + edge_pz
    
    print(f"\nEstimated per-edge Pauli rates (2 SWAPs):")
    print(f"  pX={edge_px:.6f}, pY={edge_py:.6f}, pZ={edge_pz:.6f}")
    print(f"  Total: {edge_total:.6f} ({edge_total*100:.2f}%)")
    if edge_total > 0:
        print(f"  Ratio pX:pY:pZ = {edge_px/edge_total:.2f}:{edge_py/edge_total:.2f}:{edge_pz/edge_total:.2f}")
    
    return {
        'px': edge_px,
        'py': edge_py,
        'pz': edge_pz,
        'total': edge_total,
        'sx_rates': {'px': avg_sx_px, 'py': avg_sx_py, 'pz': avg_sx_pz},
        'cz_rates': {'px': median_cz_px, 'py': median_cz_py, 'pz': median_cz_pz}
    }


# Compute ground truth with actual Pauli distribution
edge_gt = compute_edge_pauli_ground_truth(sx_pauli_rates, cz_pauli_rates)

Extracted Pauli rates from noise model:
SX gate (average across 156 qubits):
  pX=0.000109, pY=0.000109, pZ=0.000255
  Total: 0.000474
  Ratio pX:pY:pZ = 0.23:0.23:0.54

CZ gate (median across 176 edges, single-qubit marginal):
  pX=0.002823, pY=0.002823, pZ=0.002403
  Total: 0.008048
  Ratio pX:pY:pZ = 0.35:0.35:0.30

Estimated per-edge Pauli rates (2 SWAPs):
  pX=0.018247, pY=0.018247, pZ=0.017479
  Total: 0.053972 (5.40%)
  Ratio pX:pY:pZ = 0.34:0.34:0.32


## 8. Generate Ground Truth for Network Graph

In [9]:
# Load your network graph
NETWORK_GRAPH_PATH = '../SwapChainNbks/../../pickle_files/2x3_grid_network_graph.pkl'

try:
    with open(NETWORK_GRAPH_PATH, 'rb') as f:
        network_graph = pickle.load(f)
    print(f"Loaded network graph with {len(network_graph.nodes())} nodes and {len(network_graph.edges())} edges")
except FileNotFoundError:
    # Create a sample graph
    network_graph = nx.grid_2d_graph(2, 3)
    network_graph = nx.convert_node_labels_to_integers(network_graph)
    print(f"Created sample 2x3 grid graph")

print(f"Edges: {list(network_graph.edges())}")

Created sample 2x3 grid graph
Edges: [(0, 3), (0, 1), (1, 4), (1, 2), (2, 5), (3, 4), (4, 5)]


In [10]:
# Option A: Same rates for all edges (global average)
print("Option A: Global average rates (same for all edges)")
print("=" * 60)

GROUND_TRUTH_UNIFORM = {}
for edge in network_graph.edges():
    GROUND_TRUTH_UNIFORM[edge] = [edge_gt['px'], edge_gt['py'], edge_gt['pz']]
    
for edge, rates in GROUND_TRUTH_UNIFORM.items():
    print(f"  Edge {edge}: pX={rates[0]:.6f}, pY={rates[1]:.6f}, pZ={rates[2]:.6f}")

Option A: Global average rates (same for all edges)
  Edge (0, 3): pX=0.018247, pY=0.018247, pZ=0.017479
  Edge (0, 1): pX=0.018247, pY=0.018247, pZ=0.017479
  Edge (1, 4): pX=0.018247, pY=0.018247, pZ=0.017479
  Edge (1, 2): pX=0.018247, pY=0.018247, pZ=0.017479
  Edge (2, 5): pX=0.018247, pY=0.018247, pZ=0.017479
  Edge (3, 4): pX=0.018247, pY=0.018247, pZ=0.017479
  Edge (4, 5): pX=0.018247, pY=0.018247, pZ=0.017479


In [11]:
# Option B: Map logical edges to specific FakeFez physical edges
print("\nOption B: Use specific FakeFez physical edges (different per edge)")
print("=" * 60)

# See the range of CZ errors available
cz_errors_sorted = sorted(cz_pauli_rates.items(), key=lambda x: x[1]['total_error'])
print("FakeFez CZ error range:")
print(f"  Best edge:  {cz_errors_sorted[0][0]} with error {cz_errors_sorted[0][1]['total_error']:.6f}")
print(f"  Median:     {cz_errors_sorted[len(cz_errors_sorted)//2][0]} with error {cz_errors_sorted[len(cz_errors_sorted)//2][1]['total_error']:.6f}")
print(f"  Worst good: {cz_errors_sorted[-10][0]} with error {cz_errors_sorted[-10][1]['total_error']:.6f}")

# Pick different FakeFez edges spread across the error spectrum
num_edges = len(list(network_graph.edges()))
step = len(cz_errors_sorted) // (num_edges + 1)

LOGICAL_TO_PHYSICAL_EDGE = {}
for i, logical_edge in enumerate(network_graph.edges()):
    idx = min((i + 1) * step, len(cz_errors_sorted) - 20)
    physical_edge = cz_errors_sorted[idx][0]
    LOGICAL_TO_PHYSICAL_EDGE[logical_edge] = physical_edge

print(f"\nLogical to Physical edge mapping:")
for logical, physical in LOGICAL_TO_PHYSICAL_EDGE.items():
    err = cz_pauli_rates[physical]['total_error']
    print(f"  Logical {logical} -> Physical {physical} (CZ error: {err:.6f})")


Option B: Use specific FakeFez physical edges (different per edge)
FakeFez CZ error range:
  Best edge:  (22, 23) with error 0.002674
  Median:     (90, 91) with error 0.004883
  Worst good: (63, 64) with error 0.041573

Logical to Physical edge mapping:
  Logical (0, 3) -> Physical (91, 92) (CZ error: 0.003612)
  Logical (0, 1) -> Physical (85, 86) (CZ error: 0.004036)
  Logical (1, 4) -> Physical (61, 62) (CZ error: 0.004365)
  Logical (1, 2) -> Physical (90, 91) (CZ error: 0.004883)
  Logical (2, 5) -> Physical (57, 67) (CZ error: 0.005347)
  Logical (3, 4) -> Physical (112, 113) (CZ error: 0.007009)
  Logical (4, 5) -> Physical (26, 27) (CZ error: 0.012842)


In [12]:
def compute_edge_pauli_from_physical(physical_edge, sx_pauli_rates, cz_pauli_rates):
    """
    Compute Pauli rates for a logical edge using a specific FakeFez physical edge.
    """
    q1, q2 = physical_edge
    
    # Get SX errors for these specific qubits
    sx1 = sx_pauli_rates.get(q1, {'px': 0, 'py': 0, 'pz': 0})
    sx2 = sx_pauli_rates.get(q2, {'px': 0, 'py': 0, 'pz': 0})
    
    avg_sx_px = (sx1['px'] + sx2['px']) / 2
    avg_sx_py = (sx1['py'] + sx2['py']) / 2
    avg_sx_pz = (sx1['pz'] + sx2['pz']) / 2
    
    # Get CZ error for this specific edge
    cz_rates = cz_pauli_rates.get(physical_edge, cz_pauli_rates.get((physical_edge[1], physical_edge[0])))
    if cz_rates is None:
        return {'px': 0, 'py': 0, 'pz': 0, 'total': 0}
    
    diag = cz_rates['ptm_diagonal']
    
    # Extract marginal single-qubit Pauli rates from CZ
    d_X = np.mean(diag[4:8])
    d_Y = np.mean(diag[8:12])
    d_Z = np.mean(diag[12:16])
    
    cz_px = max(0, (1 - d_X) / 2)
    cz_py = max(0, (1 - d_Y) / 2)
    cz_pz = max(0, (1 - d_Z) / 2)
    
    # SWAP = 3 CZ + ~6 SX, per edge = 2 SWAPs
    edge_px = 2 * (3 * cz_px + 6 * avg_sx_px)
    edge_py = 2 * (3 * cz_py + 6 * avg_sx_py)
    edge_pz = 2 * (3 * cz_pz + 6 * avg_sx_pz)
    
    return {
        'px': edge_px,
        'py': edge_py,
        'pz': edge_pz,
        'total': edge_px + edge_py + edge_pz
    }


# Generate ground truth with per-edge physical mapping
GROUND_TRUTH_PER_EDGE = {}

print(f"Per-edge Pauli rates using physical mapping:")
print("-" * 70)
for logical_edge in network_graph.edges():
    physical_edge = LOGICAL_TO_PHYSICAL_EDGE[logical_edge]
    rates = compute_edge_pauli_from_physical(physical_edge, sx_pauli_rates, cz_pauli_rates)
    GROUND_TRUTH_PER_EDGE[logical_edge] = [rates['px'], rates['py'], rates['pz']]
    
    print(f"  {logical_edge} -> {physical_edge}: pX={rates['px']:.6f}, pY={rates['py']:.6f}, pZ={rates['pz']:.6f}, total={rates['total']:.6f}")

# Show the variation
totals = [sum(v) for v in GROUND_TRUTH_PER_EDGE.values()]
print(f"\nVariation in total error:")
print(f"  Min: {min(totals):.6f}, Max: {max(totals):.6f}, Range: {max(totals)-min(totals):.6f}")

Per-edge Pauli rates using physical mapping:
----------------------------------------------------------------------
  (0, 3) -> (91, 92): pX=0.012799, pY=0.012799, pZ=0.015922, total=0.041520
  (0, 1) -> (85, 86): pX=0.014252, pY=0.014252, pZ=0.016364, total=0.044868
  (1, 4) -> (61, 62): pX=0.018159, pY=0.018159, pZ=0.014397, total=0.050716
  (1, 2) -> (90, 91): pX=0.023050, pY=0.023050, pZ=0.013539, total=0.059639
  (2, 5) -> (57, 67): pX=0.020353, pY=0.020353, pZ=0.019892, total=0.060598
  (3, 4) -> (112, 113): pX=0.027194, pY=0.027194, pZ=0.023895, total=0.078284
  (4, 5) -> (26, 27): pX=0.042727, pY=0.042727, pZ=0.043240, total=0.128695

Variation in total error:
  Min: 0.041520, Max: 0.128695, Range: 0.087175


## 9. Save Ground Truth Files

In [13]:
OUTPUT_DIR = '../../pickle_files/syndrome_data_513/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Uniform version
with open(os.path.join(OUTPUT_DIR, 'ground_truth_uniform.pkl'), 'wb') as f:
    pickle.dump(GROUND_TRUTH_UNIFORM, f)
print(f"Saved: ground_truth_uniform.pkl (same rates for all edges)")

# Per-edge version
with open(os.path.join(OUTPUT_DIR, 'ground_truth_per_edge.pkl'), 'wb') as f:
    pickle.dump(GROUND_TRUTH_PER_EDGE, f)
print(f"Saved: ground_truth_per_edge.pkl (different rates per edge)")

# Physical mapping
with open(os.path.join(OUTPUT_DIR, 'logical_to_physical_mapping.pkl'), 'wb') as f:
    pickle.dump(LOGICAL_TO_PHYSICAL_EDGE, f)
print(f"Saved: logical_to_physical_mapping.pkl")

print(f"\nAll files saved to {OUTPUT_DIR}")

Saved: ground_truth_uniform.pkl (same rates for all edges)
Saved: ground_truth_per_edge.pkl (different rates per edge)
Saved: logical_to_physical_mapping.pkl

All files saved to ../../pickle_files/syndrome_data_513/


## Summary: Ground Truth Options

| Method | Pros | Cons |
|--------|------|------|
| **Empirical (circuit simulation)** | Captures full circuit behavior | Noisy, includes prep/measure errors |
| **Noise model average** | Fast, no simulation needed | Ignores qubit-specific variations |
| **Noise model with mapping** | Exact per-edge values | Requires topology-aware design |

### Recommendations:

1. **For quick experiments**: Use noise model average (`ground_truth_uniform.pkl`)
2. **For accurate per-edge GT**: Use mapped version (`ground_truth_per_edge.pkl`)
3. **For production**: Use the `513DataGenerationBase.ipynb` which integrates both methods